<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/10_memory_knowledge_access/notebook_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 10: Memory for Agents

This lesson explores the concept of adding **long-term memory** to agents, so they can persist and retrieve information over time. 

We’ll implement semantic, episodic, and procedural memory using the open-source mem0 library with Google's Gemini text embedding model, and a vector store that runs locally in the notebook, using ChromaDB. 


Learning Objectives:

1. Understand the different types of memory 
2. How to implement them, using the mem0 library.

> **Exercise version.** This is the exercise notebook for Lesson 10. The full solution lives in [`notebook.ipynb`](https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/10_memory_knowledge_access/notebook.ipynb) in the same folder. Attempt each exercise before checking the solutions. The three exercises build on each other: the wrappers from Exercise 1 are what Exercises 2 and 3 use to store their memories.

## 1. Setup


### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages and automatically loads your credentials from Colab Secrets (your `GOOGLE_API_KEY`, or your Vertex AI settings if you chose that option in the Course Admin lesson).

To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL;DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.


In [ ]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import site
    import subprocess
    import warnings

    # Suppress DeprecationWarning from Colab's jupyter_client (datetime.utcnow on Python 3.12)
    warnings.filterwarnings("ignore", category=DeprecationWarning, module="jupyter_client")

    # Install the course package (published from pyproject.toml) and its pinned extras
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==0.4.17",
            "nest-asyncio2",
            "mem0ai==2.0.4",
            "google-auth==2.53.0",
            "opentelemetry-api==1.42.1",
            "opentelemetry-sdk==1.42.1",
            "opentelemetry-exporter-otlp-proto-http==1.42.1",
            "opentelemetry-exporter-otlp-proto-common==1.42.1",
            "opentelemetry-proto==1.42.1",
            "jedi==0.18.2",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

In [ ]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

from utils import env

env.load(required_env_vars=["GOOGLE_API_KEY"])

### Import Key Packages

In [ ]:
import os
import re
from typing import Optional

from google import genai
from mem0 import Memory

### Initialize the Gemini Client

In [ ]:
client = genai.Client()

### Define Constants

We will use the `gemini-3.5-flash` model, which is fast and cost-effective:

In [ ]:
MODEL_ID = "gemini-3.5-flash"

### Configure mem0 (Gemini LLM + embeddings + local vector store)

Here we instantiate mem0 with:

- LLM: our existing Gemini model (`MODEL_ID = "gemini-3.5-flash"`) for the summarization/extraction of facts.
- Embeddings: Gemini's `gemini-embedding-001` (output reduced to 768 dimensions).
- Vector store:
    - ChromaDB with `MEM_BACKEND=chromadb` 

In [ ]:
MEM0_CONFIG = {
    # Use Google's gemini-embedding-001 for embeddings (output reduced to 768-dim)
    "embedder": {
        "provider": "gemini",
        "config": {
            "model": "gemini-embedding-001",
            "embedding_dims": 768,
            "api_key": os.getenv("GOOGLE_API_KEY"),
        },
    },
    # Use ChromaDB as a local, in-notebook vector store
    "vector_store": {
        "provider": "chroma",
        "config": {
            "collection_name": "lesson9_memories",
            "path": "/tmp/chroma_mem0",
        },
    },
    "llm": {
        "provider": "gemini",
        "config": {
            "model": MODEL_ID,
            "api_key": os.getenv("GOOGLE_API_KEY"),
        },
    },
}

memory = Memory.from_config(MEM0_CONFIG)
MEM_USER_ID = "lesson9_notebook_student"
memory.delete_all(user_id=MEM_USER_ID)
print("✅ Mem0 ready (Gemini embeddings + local Chroma).")

### Helper functions: add/search for memories

A small wrapper layer around mem0 to:

- Save a string memory and tag it with a category ("semantic", "episodic", "procedure") plus any extra metadata.
    - `mem_add_text` stores verbatim text with infer=False (no LLM fact extraction triggered by mem0). It also changes and all metadata values to primitives (str | int | float | bool | None) since mem0 requires primitive types.

- Search memories and (optionally) filter by category client-side.
    - `mem_search` calls memory.search(...) and then inspects each hit’s metadata to filter.

### Exercise 1: Implement the memory wrappers

Everything in this lesson flows through two small functions: one that saves a tagged memory, one that searches with an optional category filter. They are your interface to mem0.

**Learning goal:** Store and retrieve categorized memories through the mem0 API.

**What you need to implement:**

1. `mem_add_text`: build a metadata dict that starts with the category, add every extra keyword argument (coercing values that are not str/int/float/bool/None to strings, mem0 only accepts primitives), then add the text under `MEM_USER_ID` with mem0's LLM fact extraction disabled (its `infer` flag), and return a short confirmation string mentioning the category
2. `mem_search`: search mem0 with the query, a filters dict scoping to `MEM_USER_ID`, and the limit passed through `top_k` (fall back to an empty dict if the call returns nothing), take the list under the result's `"results"` key, keep only hits whose metadata category matches when a category is given, and return the list

**Key concepts:**

- `memory.add(...)` accepts `user_id`, `metadata`, and an `infer` flag that controls whether mem0 runs LLM extraction over the text
- `memory.search(...)` returns a dict whose `"results"` key holds the hit dicts, each with `"memory"` and `"metadata"` entries
- Category filtering happens client-side here, by inspecting each hit's metadata

**Expected output:** the semantic-memory cell below prints a confirmation per fact (e.g. `Saved semantic memory.`) and later search cells return hits with `memory` and `metadata` fields.

**Implementation hints:**

- `(r.get("metadata") or {})` is the safe way to read metadata that might be missing
- The raw-search demo cell a bit further down indexes `results["results"][0]` directly, it will raise an IndexError until your `mem_add_text` actually stores the facts (re-run the facts cell after implementing)

In [ ]:
# === Exercise cell: fill in the gaps below ===


def mem_add_text(text: str, category: str = "semantic", **meta) -> str:
    """Add a single text memory. No LLM is used for extraction or summarization.

    Steps to complete:
    1. Build a metadata dict containing the category
    2. Copy the extra keyword arguments into it, converting any value that is
       not a primitive (str/int/float/bool/None) to a string
    3. Add the text to mem0 under MEM_USER_ID with the metadata, disabling
       mem0's LLM fact extraction
    4. Return a short confirmation string that mentions the category
    """
    # Your implementation goes here

    return ""  # Replace with the confirmation string


def mem_search(query: str, limit: int = 5, category: Optional[str] = None) -> list[dict]:
    """
    Category-aware search wrapper.
    Returns the full result dicts so we can inspect metadata.

    Steps to complete:
    1. Search mem0 with the query, a filters dict scoping to MEM_USER_ID,
       and the limit (fall back to an empty dict if the call returns nothing)
    2. Take the list stored under the result's "results" key
    3. If a category was given, keep only hits whose metadata category matches
    4. Return the list
    """
    # Your implementation goes here

    return []  # Replace with the list of matching result dicts

## 2. Semantic memory example (facts as atomic strings)

**Goal**: We show semantic memory as “facts & preferences” stored as short, individual strings.

- We insert a few example facts (e.g., “User has a dog named George”).

- Then we search with a natural query (e.g., “brother job”) and see the relevant fact returned.

In [ ]:
facts: list[str] = [
    "User prefers vegetarian meals.",
    "User has a dog named George.",
    "User is allergic to gluten.",
    "User's brother is named Mark and is a software engineer.",
]
for f in facts:
    print(mem_add_text(f, category="semantic"))

print(f"Added {len(facts)} semantic memories.")

In [ ]:
# Search for a specific fact
results = memory.search("brother job", filters={"user_id": MEM_USER_ID}, top_k=1)
# We print the memory string
print(results["results"][0]["memory"])
# We print the whole dict that contains the memory
print(results["results"][0])

### Validation check - run this after your implementation

Uncomment the cell below and run it after implementing Exercise 1 and re-running the facts cell above. It performs one embedding-backed search.

In [ ]:
# # Validation: check your Exercise 1 implementation
# try:
#     _msg = mem_add_text("Validation probe: user enjoys hiking.", category="semantic", probe=True)
#     assert isinstance(_msg, str) and "semantic" in _msg, "❌ mem_add_text should return a confirmation mentioning the category."
#     _hits = mem_search("dog named George", limit=3, category="semantic")
#     assert _hits, "❌ No semantic hits found. Did you re-run the facts cell after implementing?"
#     assert any("George" in h.get("memory", "") for h in _hits), "❌ Expected the dog fact among the top hits."
#     assert all((h.get("metadata") or {}).get("category") == "semantic" for h in _hits), "❌ The category filter let a non-semantic hit through."
#     print("✅ All checks passed! Your memory wrappers store and retrieve correctly.")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: check that metadata carries the category and that the search reads the 'results' list.")

## 3. Episodic memory example (summarize 3–4 turns → one episode)

**Goal**: Demonstrate episodic memory (experiences & history).

- We create a short 3–4 turn exchange between user and assistant.

- We ask the LLM to produce a concise episode summary (1–2 sentences) and save it under category="episodic".

- Finally, we run a semantic search (e.g., “deadline stress”) to retrieve that episode, we print the memory along with its creation timestamp.

This example show how an agent can compress transient chat into a single durable “moment.”

Since mem0 by default creates a created_at timestamp, we have the possibility to use it to sort and filter memories.
It would then be possible to answer questions like "What did we talk about last week?"

### Exercise 2: Compress a dialogue into an episodic memory

Episodic memory stores experiences, not facts. The move here: take a short multi-turn exchange and have the LLM write the one durable "moment" worth keeping.

**Learning goal:** Use an LLM to summarize transient conversation into a single episode string.

**What you need to implement:**

1. Write `episodic_prompt`: ask the model to summarize the given 3-4 turns as one concise episode of 1-2 sentences, keeping salient details and tone, with the `dialogue` embedded in the prompt
2. Call the model with your prompt
3. Assign the stripped response text to `episode`

**Key concepts:**

- The prompt is the design decision here: what to keep (deadline, Friday, night-owl preference) is decided by your instructions
- An f-string can embed the `dialogue` list directly, the model reads the role/content structure just fine

**Expected output:** one or two sentences, e.g. a summary mentioning the user's stress about a Friday project deadline, the testing blocker, and the plan to split testing.

**Implementation hints:**

- Complete this exercise (and Exercise 1) before running the save-and-search cell below, it stores whatever `episode` holds
- The smoke test makes one API call once the prompt is in place, none while it is `None`

In [ ]:
# === Exercise cell: fill in the gaps below ===

# A short 4-turn exchange we want to compress into one "episode"
dialogue = [
    {"role": "user", "content": "I'm stressed about my project deadline on Friday."},
    {"role": "assistant", "content": "I’m here to help—what’s the blocker?"},
    {"role": "user", "content": "Mainly testing. I also prefer working at night."},
    {"role": "assistant", "content": "Okay, we can split testing into two sessions."},
]

# Steps to complete:
# 1. Write a prompt that asks for a concise 1-2 sentence episode summary of the
#    dialogue, keeping salient details and tone
# 2. Call the model with your prompt
# 3. Store the stripped response text in `episode`

episodic_prompt = None  # TODO 1: author the summarization prompt
episode = ""  # TODO 2 and 3: call the model and keep the stripped text

# Your implementation goes here

# Smoke test
if episodic_prompt is None:
    print("(episodic_prompt not written yet, complete the TODOs above)")
else:
    print(episode if episode else "(episode is empty, check your model call)")

In [ ]:
print(
    mem_add_text(
        episode,
        category="episodic",
        summarized=True,
        turns=4,
    )
)

print("\nSearch --> 'deadline stress'\n")
hits = mem_search("deadline stress", limit=1, category="episodic")
for h in hits:
    print(f"{h['memory']}\n")
    print(h)

## 4. Procedural memory example (learn & “run” a skill)

**Goal**: Demonstrate procedural memory (skills & workflows).

- We teach the agent a small procedure (e.g., monthly_report) by saving ordered steps in a single text block under category="procedure".

- We retrieve the procedure and parse the numbered steps to simulate “running” it.

This example shows how agents can learn reusable playbooks and trigger them later by name.

### Exercise 3: Teach the agent a procedure

Procedural memory stores playbooks: named, ordered steps an agent can retrieve and follow later. The storage format is the exercise, retrieval is already solved below.

**Learning goal:** Encode a reusable multi-step skill as a single retrievable memory.

**What you need to implement:**

1. Build `procedure_text` as one text block: a first line naming the procedure, a `Steps:` line, then the steps numbered starting at 1, one per line

**Key concepts:**

- `enumerate(steps)` pairs each step with its index for numbering
- Storing the whole procedure as one block (rather than one memory per step) keeps the steps ordered and retrievable together

**Expected output:** the retrieval cell below prints the stored block:

```text
Procedure: monthly_report
Steps:
1. Query sales DB for the last 30 days.
2. Summarize top 5 insights.
3. Ask user whether to email or display.
```

**Implementation hints:**

- A join over a generator of numbered lines keeps this to two short lines of code
- The save is already wired below your TODO, it runs once `procedure_text` is non-empty (and needs Exercise 1 done)

In [ ]:
# === Exercise cell: fill in the gaps below ===

procedure_name = "monthly_report"
steps = [
    "Query sales DB for the last 30 days.",
    "Summarize top 5 insights.",
    "Ask user whether to email or display.",
]

# Step to complete: build the procedure text block described in the briefing
# (name line, "Steps:" line, then the numbered steps, one per line)
procedure_text = ""  # TODO: compose the block from procedure_name and steps

# Your implementation goes here

# Save the procedure (runs once procedure_text is filled in)
if procedure_text:
    mem_add_text(procedure_text, category="procedure", procedure_name=procedure_name)
    print(f"Learned procedure: {procedure_name}")
else:
    print("(procedure_text is empty, complete the TODO above)")

In [ ]:
# Retrieve the procedure by name
results = mem_search("how to create a monthly report", category="procedure", limit=1)
if results:
    print(results[0]["memory"])

## Stretch challenges

Want to go further? Try these on your own:

1. Use the `created_at` timestamp mem0 attaches to every memory to answer "what did we talk about today?": search episodic memories, then filter the hits by date client-side.
2. Write a `run_procedure(name)` helper that retrieves a procedure by name, parses the numbered steps with a regular expression (the `re` import is already there), and prints each step as if executing it.
3. Store the raw 4-turn dialogue again with mem0's fact extraction enabled instead of disabled, then compare what mem0 extracted on its own against your Exercise 2 episode summary.